# Extracting Structured Invoice Data With the OpenAI API

**Stephanie Nord**  
**DSC 670: Generative Artificial Intelligence**  
**Problem 4: Extraction**

## Purpose

The goal of this notebook is to use the OpenAI API in two distinct stages. First, the notebook sends a PDF invoice to a multimodal language model with instructions to extract the document’s text without interpreting or correcting it. Second, it sends the extracted text to the model and requests a structured JSON representation. The notebook then checks selected values against values taken from the original invoice.

Text extraction and JSON conversion are separated because the two stages have different goals. The first stage focuses on accurately preserving the document’s original content, while the second focuses on organizing that information using consistent field names and data types. Keeping the stages separate also makes it easier to determine whether an error occurred during text extraction or conversion to JSON.

## Setup and Security

The API key is entered with `getpass`, which hides it while it is typed and prevents it from appearing in the notebook. I do not save the key in a code cell or commit it to a repository. The invoice PDF should be placed in the same folder as this notebook and should retain the filename `dsc-670-exercise-invoice.pdf`.

The notebook uses the OpenAI Responses API because it accepts PDF files as model input. It also uses a Pydantic schema during the second request so the response is returned in a predictable structure rather than as loosely formatted prose.

In [1]:
import json
import os
from getpass import getpass
from pathlib import Path
from typing import Optional

from openai import OpenAI
from pydantic import BaseModel, Field

if not os.getenv("OPENAI_API_KEY"):
    os.environ["OPENAI_API_KEY"] = getpass("Enter your OpenAI API key: ")

client = OpenAI()
MODEL = "gpt-4.1-mini"

invoice_path = Path("dsc-670-exercise-invoice.pdf")
if not invoice_path.exists():
    raise FileNotFoundError(
        "Place dsc-670-exercise-invoice.pdf in the same folder as this notebook."
    )

print(f"Invoice found: {invoice_path.resolve()}")
print(f"File size: {invoice_path.stat().st_size:,} bytes")
print(f"Model: {MODEL}")

Enter your OpenAI API key:  ········


Invoice found: C:\Users\sfnor\Projects\generative_ai\notebooks\dsc-670-exercise-invoice.pdf
File size: 260,587 bytes
Model: gpt-4.1-mini


## Stage 1: Extract the Invoice Text

Before running the extraction, I anticipated the invoice to be more intuitive and straightforward. Instead it is a pdf that is a single page, and it has labeled fields. The table layout creates a challenge because the model must be able to associate quantities, descriptions, unit prices, and amounts across columns. The bottom of the invoice also has a negative balance because there is a one thousand dollar credit which is much more than the six dollar subtotal. I instructed the model to transcribe this information rather than to correct the document so that an unusual situation can still preserve proper data.

In [9]:
uploaded_file = client.files.create(
    file=invoice_path.open("rb"),
    purpose="user_data",
)

print(f"Uploaded file ID: {uploaded_file.id}")

Uploaded file ID: file-5W3RSwkszDqFmKzku5sQMP


In [10]:
transcription_prompt = '''Transcribe all readable text from this invoice.
Preserve the relationships among labels, values, table columns, and line items.
Preserve punctuation, percentage signs, currency amounts, parentheses, and placeholder text.
Do not summarize, calculate, correct, or infer missing information.
Represent blank cells as blank rather than inventing values.
Return only the transcription in readable plain text.'''

text_response = client.responses.create(
    model=MODEL,
    input=[
        {
            "role": "user",
            "content": [
                {"type": "input_text", "text": transcription_prompt},
                {
                    "type": "input_file",
                    "file_id": uploaded_file.id,
                    "detail": "high",
                },
            ],
        }
    ],
)

extracted_text = text_response.output_text
print(extracted_text)

321 Avenue A  
Portland, OR 12345  
Phone: (206) 555-1163  
Fax: (206) 555-1164  
someone@websitegoeshere.com  

Date: 6/28/2024  
Invoice # 1111  
For PO # 123456  

Quantity  Description       Unit price  Amount  Discount applied  
1         Item Number 1     $ 2.00      2.00    $  
1         Item Number 2     $ 2.00      2.00    $  
1         Item Number 3     $ 2.00      2.00    $  
$ -  
$ -  
$ -  
$ -  
$ -  
$ -  
$ -  
$ -  

Subtotal                          6.00     $  
Credit                         $ 1,000.00  
Tax                             9.80%  
Additional discount             12%  
Balance due                   (994.20)  $  

Company Name  
INVOICE  

If you have any questions concerning this invoice, contact  
<Name> at <phone or email>.  
Make all checks payable to <Company name>.  
Thank you for your business!  

Bill To:  
Natasha Jones  
Central Beauty  
123 Main St.  
Manhattan, NY 98765  
(321) 555-1234  

% discount 10%  

Items over this amount qualify for a

### Commentary on the Text Extraction

I compared the extracted text with the original PDF and verified the invoice number, purchase order number, three line items, subtotal, credit, percentages, and negative balance. The parentheses around the balance due indicate a negative amount. The model also preserved placeholder text such as “Company Name.” Although the order and spacing differed from the original invoice, the information I checked was retained.

## Stage 2: Convert the Extracted Text to Structured JSON

For the second stage, the notebook uses a predefined structure to organize the extracted text into JSON. Missing or uncertain values are represented as null so the model does not have to guess. Monetary values are stored as numbers without dollar signs, and percentages are stored as numbers, such as 9.8 for 9.8%. The balance shown in parentheses is converted to the negative number -994.2, which is equivalent to -994.20.

In [11]:
class Party(BaseModel):
    name: Optional[str] = None
    company: Optional[str] = None
    street: Optional[str] = None
    city: Optional[str] = None
    state: Optional[str] = None
    postal_code: Optional[str] = None
    phone: Optional[str] = None
    fax: Optional[str] = None
    email: Optional[str] = None


class LineItem(BaseModel):
    quantity: Optional[float] = None
    description: Optional[str] = None
    unit_price: Optional[float] = None
    amount: Optional[float] = None
    discount_applied: Optional[float] = None


class Invoice(BaseModel):
    vendor: Party
    bill_to: Party
    invoice_date: Optional[str] = Field(
        default=None, description="Date in YYYY-MM-DD format when possible"
    )
    invoice_number: Optional[str] = None
    purchase_order_number: Optional[str] = None
    discount_qualification_amount: Optional[float] = None
    qualifying_discount_percent: Optional[float] = None
    line_items: list[LineItem]
    subtotal: Optional[float] = None
    credit: Optional[float] = None
    tax_percent: Optional[float] = None
    additional_discount_percent: Optional[float] = None
    balance_due: Optional[float] = None
    currency: Optional[str] = None
    payment_instructions: Optional[str] = None
    notes: list[str] = []

In [12]:
json_prompt = f'''Convert the invoice transcription below into the provided schema.

Rules:
- Use only information present in the transcription.
- Use null for missing or uncertain values; never guess.
- Exclude completely blank line-item rows.
- Store money as numbers without currency symbols or commas.
- Store percentages as numeric percentage values, so 9.80% becomes 9.80.
- Interpret a monetary value in parentheses as negative, so ($994.20) becomes -994.20.
- Preserve placeholder text such as Company Name when it is visibly present.
- Do not recalculate or silently correct the invoice.

INVOICE TRANSCRIPTION:
{extracted_text}'''

structured_response = client.responses.parse(
    model=MODEL,
    input=[
        {
            "role": "system",
            "content": "You extract invoice data faithfully into the requested structure.",
        },
        {"role": "user", "content": json_prompt},
    ],
    text_format=Invoice,
)

invoice = structured_response.output_parsed
invoice_json = invoice.model_dump()

print(json.dumps(invoice_json, indent=2))

{
  "vendor": {
    "name": null,
    "company": "Company Name",
    "street": "321 Avenue A",
    "city": "Portland",
    "state": "OR",
    "postal_code": "12345",
    "phone": "(206) 555-1163",
    "fax": "(206) 555-1164",
    "email": "someone@websitegoeshere.com"
  },
  "bill_to": {
    "name": "Natasha Jones",
    "company": "Central Beauty",
    "street": "123 Main St.",
    "city": "Manhattan",
    "state": "NY",
    "postal_code": "98765",
    "phone": "(321) 555-1234",
    "fax": null,
    "email": null
  },
  "invoice_date": "6/28/2024",
  "invoice_number": "1111",
  "purchase_order_number": "123456",
  "discount_qualification_amount": 100.0,
  "qualifying_discount_percent": 10.0,
  "line_items": [
    {
      "quantity": 1.0,
      "description": "Item Number 1",
      "unit_price": 2.0,
      "amount": 2.0,
      "discount_applied": null
    },
    {
      "quantity": 1.0,
      "description": "Item Number 2",
      "unit_price": 2.0,
      "amount": 2.0,
      "discount_a

### Save the JSON Output

Saving the result creates a separate JSON file that can be opened independently of the notebook. This also confirms that the final structure can be serialized as valid JSON.


In [13]:
json_output_path = Path("structured_invoice.json")
json_output_path.write_text(
    json.dumps(invoice_json, indent=2),
    encoding="utf-8",
)

print(f"Saved JSON to: {json_output_path.resolve()}")

Saved JSON to: C:\Users\sfnor\Projects\generative_ai\notebooks\structured_invoice.json


## Validation

Valid JSON does not guarantee accurate information. The notebook checks selected values against the original invoice and confirms that the three line-item amounts add up to the subtotal. These checks focus on important details, including the invoice number, credit, percentages, and negative balance. Passing these checks supports the accuracy of those values, but it does not verify every field in the output.

In [14]:
checks = {
    "invoice_number": invoice.invoice_number == "1111",
    "purchase_order_number": invoice.purchase_order_number == "123456",
    "three_line_items": len(invoice.line_items) == 3,
    "subtotal": invoice.subtotal == 6.00,
    "credit": invoice.credit == 1000.00,
    "tax_percent": invoice.tax_percent == 9.80,
    "additional_discount_percent": invoice.additional_discount_percent == 12.0,
    "negative_balance_due": invoice.balance_due == -994.20,
}

line_item_total = sum(item.amount or 0 for item in invoice.line_items)
checks["line_items_equal_subtotal"] = round(line_item_total, 2) == invoice.subtotal

for check_name, passed in checks.items():
    print(f"{check_name:32} {'PASS' if passed else 'REVIEW'}")

print(f"\nLine-item total: ${line_item_total:,.2f}")
print(f"Invoice subtotal: ${invoice.subtotal:,.2f}")

invoice_number                   PASS
purchase_order_number            PASS
three_line_items                 PASS
subtotal                         PASS
credit                           PASS
tax_percent                      PASS
additional_discount_percent      PASS
negative_balance_due             PASS
line_items_equal_subtotal        PASS

Line-item total: $6.00
Invoice subtotal: $6.00


## Evaluation of the Results

The two-stage process made it easier to compare the extracted text with the structured JSON. The output contained three line items, each with a quantity of 1, a unit price of 2.00 dollars, and an amount of 2.00 dollars. These amounts matched the subtotal of 6.00 dollars. The model also preserved the credit of 1,000.00 dollars and converted the balance shown in parentheses to the negative number `-994.2`.

The JSON output used `null` for blank line-item discount fields. This distinction matters because a blank field does not necessarily mean that the discount was zero. The model also retained placeholder text such as “Company Name” without replacing it with invented information.

All nine automated validation checks passed. I also compared the selected information with the original invoice and verified those values. However, the checks did not cover every field or formatting requirement. For example, the date remained `6/28/2024` even though the schema description requested `YYYY-MM-DD` when possible. The date was accurate, but its format did not follow that instruction.

These results show that the model successfully converted the invoice into structured data for the fields reviewed. They also show why a readable output and passing checks are not enough to establish complete accuracy. Human review remains necessary to identify issues that automated checks do not cover.

While the assignment is straightforward, I think that next time I would expand on the validation to check the date format as well as each item's description, quantity, and unit price. The current checks confirmed several important values but the date format issue shows that passing every check does not mean every instruction was followed. Checking individual fields would help improve confidence rather than just checking the subtotal.

## Cleanup

The uploaded API file is deleted after the work is complete because it is no longer needed for this assignment. The local PDF, notebook, and generated JSON file are unaffected.


In [15]:
client.files.delete(uploaded_file.id)
print("Uploaded API file deleted.")

Uploaded API file deleted.


## References

OpenAI. (n.d.). *File inputs*. OpenAI API documentation. https://developers.openai.com/api/docs/guides/file-inputs

OpenAI. (n.d.). *Structured model outputs*. OpenAI API documentation. https://developers.openai.com/api/docs/guides/structured-outputs